In [2]:
from langchain.messages import HumanMessage,AIMessage,SystemMessage
from langchain.agents import create_agent
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START,END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import add_messages
from tavily import TavilyClient
from langchain.tools import tool
from typing import TypedDict, Annotated
from langgraph.prebuilt import ToolNode, tools_condition

In [3]:
load_dotenv()

True

In [4]:
llm = init_chat_model(model="google_genai:gemini-3.6-flash")

In [5]:
tavily_client = TavilyClient()

@tool
def tavily_search(query: str) -> dict:
    """Search the web for up-to-date information, news, facts, and answers."""
    return tavily_client.search(query=query)


In [6]:
#custom fucntion
@tool
def multiply(a:int, b:int) -> int:
    """multiply a and b
    
    args:
    a: first int
    b: second int
    
    returns:
    int: output int"""
    return a*b

In [7]:
tools = [tavily_search, multiply]
llm_with_tools = llm.bind_tools(tools)
 

In [8]:
llm_with_tools 

_ChatModelBinding(bound=ChatGoogleGenerativeAI(output_version=None, profile={}, google_api_key=SecretStr('**********'), location=None, model='gemini-3.6-flash', temperature=1.0, client=<google.genai.client.Client object at 0x00000223ECD58110>, default_metadata=(), model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'tavily_search', 'description': 'Search the web for up-to-date information, news, facts, and answers.', 'parameters': {'properties': {'query': {'type': 'string'}}, 'required': ['query'], 'type': 'object'}}}, {'type': 'function', 'function': {'name': 'multiply', 'description': 'multiply a and b\n\n    args:\n    a: first int\n    b: second int\n\n    returns:\n    int: output int', 'parameters': {'properties': {'a': {'type': 'integer'}, 'b': {'type': 'integer'}}, 'required': ['a', 'b'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [9]:
class State(TypedDict):
    messages : Annotated[list, add_messages]

In [10]:
#node functionality
def tool_calling_llm(state:State):
    #we are giving this system prompt so that the model does not loop back in aimessage and tool call which it sometimes does which burns token limit
    system = SystemMessage(content="You are a helpful assistant. Use tools when needed but avoid redundant searches. One search should be sufficient for most questions.")
    return {"messages":[llm_with_tools.invoke([system] + state['messages'])]}

In [11]:
#graph
builder =StateGraph(State)

In [12]:
#nodes
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools",ToolNode(tools))

In [13]:
#edges
builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    #if the latest message from assistant is a tool call -> tool_condition results to tools
    #If the latest message from assitant is not a tool call -> tool_conditions routes to END"""
    tools_condition
)
builder.add_edge("tools",END)

In [14]:
graph = builder.compile()

In [15]:
graph.invoke({"messages":"what is the recent news about mayor mamdani?"})

KeyboardInterrupt: 

In [ ]:
response = graph.invoke({"messages":"what is the recent news about mayor mamdani?"})

In [ ]:
response['messages'][-1].content

[{'type': 'text',
  'text': '**Zohran Mamdani**, a Democratic Socialist and member of the New York State Assembly representing District 36 (Astoria, Queens), launched a high-profile campaign for **Mayor of New York City** in the 2025 election.\n\nHere are the key points regarding his candidacy, background, and platform:\n\n---\n\n### 1. Mayoral Candidacy & Announcement\n* **Official Entry:** Campaigning as a Democratic Socialist with the backing of the NYC chapter of the Democratic Socialists of America (NYC-DSA).\n* **The Field:** Running against incumbent Mayor Eric Adams in the Democratic primary, alongside other high-profile challengers including NYC Comptroller Brad Lander and former Governor Andrew Cuomo.\n\n---\n\n### 2. Core Campaign Platform\nMamdani’s mayoral platform focuses on affordability, public services, and cost-of-living relief for working-class New Yorkers:\n* **"Freeze the Rent":** Pushing for rent freezes on rent-stabilized apartments across the city.\n* **"Fast & 

In [ ]:
graph.invoke({"messages":"WHO IS THE CURRENT MAYOR OF NEW YORK"})

{'messages': [HumanMessage(content='WHO IS THE CURRENT MAYOR OF NEW YORK', additional_kwargs={}, response_metadata={}, id='45c47b44-ac90-4359-8636-74d1169883dc'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "current Mayor of New York City"}'}, '__gemini_function_call_thought_signatures__': {'call_481299': 'Ev0CCvoCARFNMg+OROyJibvs4NL2cdi6gELfukjwakmz3MIr8gNhW+Wj8Lx7u+ScHJdSDT9f/2j6ebt8ZmUJKPHdhr6p8yNPPDlgIr4/L37/O51Zl6iF+X+AdbVKsVDmyQSE6WNhSnBqXRIIko6J0tf+RLxaRMVYERnpENAuWefD4MWkkLue4ccgDvPhlTShF0O+xUUVqY3KU5cRjP8B54rBR/caZ20eSL4W05mNsD2/FMIUWjBZLCadoVwftjbOrCSJC3bF/00HtjlOqtZun5JQHhC2+64AoeBh17FkATnWkW7iTBey9a99o9EpW0dnsvi5f9n3yhw9akYjFGR4vlwNCKxJ/RLd2v74Cx2td8j5+NapnCFAyKMoYCdr16q4YSmdNWvDE7f69cjA7mVNGPdztqM4N/FpUdpwjoTzoAWBBCk+PjHZv0B8EsKZLlCJjIrob6JP4hUe0nILtFC0Z0GrmDlrLgRWXwOZ48ePF052FosyOtIKx5U1tBJtPyyE'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.6-flash', 'safety_ratings': [], 'model_pr

In [ ]:
graph.invoke({"messages":"what is current time in san francisco?"})

{'messages': [HumanMessage(content='what is current time in san francisco?', additional_kwargs={}, response_metadata={}, id='cca38dfb-c7d5-4b17-b86a-3ccebd1c518a'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search', 'arguments': '{"query": "current time in San Francisco"}'}, '__gemini_function_call_thought_signatures__': {'call_424825': 'EsIDCr8DARFNMg8toTp4nygaCtbggdkwj0gEpW00BLECBiQu8EPUARcG5uNPpk5UMqLYwAplxsRBODQRXLQLZBHWddZbkOdqKu2Gg486Mv/tm63+XeHMGUn5C41/3l3AkYqstYDqAWmvvbwdSMCxDCVFzdKjTRS2gMO0u/gYSboLN9P6qFMDbJYZ8FHUhLeQGZWRfV7d7sdy7/qni6m3rEMulg5pyw4TO2Rw6lz63LmzRvEQS+ohuspFxwTIwiJvAtKNw6o/g2VFs3yVxrzOjyXika69Yvw05CLMmgTZdpb+keTHZ0PYH3IGrQc86tJTonIrKet3kYwa5Fk6A+YwcNAXLzOgTlrpS1Zt0hmzdwolvPr9d70yqB/4/M2pOxtvTXSLQ1qDPWE1blkwQ7aff8OmRnOrclQ8QOuVq4LY5WRcMlISDfNqps7eiBEoPZxJSWcl8l0Npst7rnblK3QDNMzizlLdvCbWjFBPlFDv2SM8/TLlGjKWIuY23fFGKGottdtRw8zp77ObgdEdydAralmracMamlfEeKiFj/Ga3mZgMIfV5wkaO3oBuSwonhmMrFDvJuzMi2H3eV9ojAX9xkpBGJWn'}}, response_metadat

In [ ]:
graph.invoke({"messages":"what is current time in san francisco?"})

ChatGoogleGenerativeAIError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash\nPlease retry in 36.856873854s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT